# 01 · Baseline, toán học và kiểm tra gradient

Từ repo của Nhựt đến một baseline có thể kiểm chứng.

## Mở trên Colab hoặc Jupyter

Đây là notebook hướng dẫn tái lập, các cell chưa được đánh dấu là đã thực thi. Kết quả huấn luyện thật có sẵn trong `artifacts/`.

**Colab:** tải ZIP đầy đủ lên Files, giải nén vào `/content/` để có `/content/Neural_Observatory`. Có thể dùng `import zipfile; zipfile.ZipFile('/content/Neural_Observatory.zip').extractall('/content')` với ZIP được bàn giao. Chọn CPU; bật T4 không tăng tốc NumPy tự động. Nếu cần dependency, chạy `%pip install -r /content/Neural_Observatory/requirements.txt`.

**Jupyter:** mở notebook từ `notebooks/` sau khi cài `requirements.txt`. Chạy các cell từ trên xuống.


In [ ]:
from pathlib import Path
import os, sys, subprocess
# Local: launch notebook from the extracted project or its notebooks/ folder.
# Colab: extract Neural_Observatory.zip into /content first (see instructions).
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/Neural_Observatory')]
project = next((p for p in candidates if (p/'observatory/engine.py').exists()), None)
assert project is not None, 'Extract the full project ZIP first and set project to its root.'
os.chdir(project)
sys.path.insert(0, str(project))
print('Project:', project)


## Vì sao cần sửa?

Repo gốc cập nhật lớp ra trước khi tính gradient lớp ẩn và chưa giữ tập test độc lập. Bản mới tính mọi gradient rồi mới cập nhật. Forward giữ quy ước W[out,in]. Loss cuối là cross-entropy với softmax ổn định.

In [ ]:
from observatory.reference import ReferenceMLP
m = ReferenceMLP((3, 4, 2), 'sigmoid', 42)
x, y = [0.3, -0.7, 0.4], 1
loss, dw, db = m.loss_and_grad(x, y)
print('Loss:', loss, 'probabilities:', m.forward(x)[0][-1])

## Đạo hàm số

Thay một trọng số bởi w+ε và w−ε; độ dốc xấp xỉ là (L(w+ε)−L(w−ε))/(2ε). So sánh với backprop. Mạng nhỏ dùng double precision để giảm sai số số học.

In [ ]:
eps = 1e-5
old = m.weights[0][0][0]
m.weights[0][0][0] = old + eps
plus = m.loss_and_grad(x,y)[0]
m.weights[0][0][0] = old - eps
minus = m.loss_and_grad(x,y)[0]
m.weights[0][0][0] = old
print('Numeric:', (plus-minus)/(2*eps), 'Backprop:', dw[0][0][0])
subprocess.run([sys.executable,'main.py','test'],check=True)

## Chạy thật bằng Python chuẩn

Lệnh -S bỏ site-packages. 1.024 ảnh train, 256 validation, 2 epoch: bản chứng minh end-to-end riêng, không phải benchmark đầy đủ.

In [ ]:
subprocess.run([sys.executable,'-S','scripts/train_pure.py','--samples','1024','--epochs','2'],check=True)

## Tự kiểm tra hiểu bài

1. Vì sao không cập nhật W trước khi tính hết gradient?
2. Vì sao softmax trừ max(logits)?
3. Vì sao accuracy trên dữ liệu đang train không phải test accuracy?